# Resultados por película, género e idioma

Combina las películas estrenadas desde 2000 con sus idiomas y géneros y guarda el resultado en `gold/results_movie_genre_language`. La tabla se registra después en `99. create_gold_tables`.

### Configuración

Carga las rutas de almacenamiento.

In [0]:
%run "../includes/configuration"

### Lectura de fuentes Silver

Lee desde Silver (Parquet): `movies`, `languages`, `movies_languages`, `genres` y `movie_genre`.

In [0]:
movies_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movies")
    
)

In [0]:
languages_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/languages")
)

In [0]:
movie_languages_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movies_languages")
)

In [0]:
genres_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/genres")
)

In [0]:
movies_genres_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movie_genre")
)

In [0]:
from pyspark.sql.functions import col, desc, asc, current_timestamp

### Construcción del resultado

Filtra películas desde 2000, las une con idiomas y géneros y agrega `created_date`.

In [0]:
results_movie_genre_language_df = (
    movies_df.filter(col("year_release_date") >= 2000)
        .withColumn("created_date", current_timestamp())
        .join(movie_languages_df, 
              movies_df["movie_id"] == movie_languages_df["movie_id"], "inner")
        .join(languages_df,
              movie_languages_df["language_id"] == languages_df["language_id"], "inner")
        .join(movies_genres_df, 
              movies_df["movie_id"] == movies_genres_df["movie_id"], "inner")
        .join(genres_df,
              movies_genres_df["genre_id"] == genres_df["genre_id"], "inner")
).select(
    movies_df["title"], movies_df["duration_time"], movies_df["release_date"], movies_df["vote_average"],
    languages_df["language_name"],
    genres_df["genre_name"],
    col("created_date")
).orderBy(
    col("release_date").asc()
)

In [0]:
display(results_movie_genre_language_df)

### Escritura en Gold

Guarda el resultado como Parquet en `gold/results_movie_genre_language` (modo `overwrite`).

In [0]:
(
    results_movie_genre_language_df
    .write
        .mode("overwrite")
        .format("parquet")
        .save(f"{gold_folder_path}/results_movie_genre_language")
)